In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

import joblib

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")
df.head()

,diseases,anxiety_and_nervousness,depression,shortness_of_breath,depressive_or_psychotic_symptoms,sharp_chest_pain,dizziness,insomnia,abnormal_involuntary_movements,chest_tightness,...,redness_in_or_around_nose,wrinkles_on_skin,foot_or_toe_weakness,hand_or_finger_cramps_or_spasms,back_stiffness_or_tightness,wrist_lump_or_mass,skin_pain,low_urine_output,sore_in_nose,ankle_weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [3]:
df.shape

(189640, 329)

In [4]:
X = df.drop(columns=["diseases"])
y = df["diseases"]

In [5]:
X

,anxiety_and_nervousness,depression,shortness_of_breath,depressive_or_psychotic_symptoms,sharp_chest_pain,dizziness,insomnia,abnormal_involuntary_movements,chest_tightness,palpitations,...,redness_in_or_around_nose,wrinkles_on_skin,foot_or_toe_weakness,hand_or_finger_cramps_or_spasms,back_stiffness_or_tightness,wrist_lump_or_mass,skin_pain,low_urine_output,sore_in_nose,ankle_weakness
0,1,0,1,1,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,1,0,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,1,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,0,0,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189635,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
189636,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
189637,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
189638,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
y

0                 panic disorder
1                 panic disorder
2                 panic disorder
3                 panic disorder
4                 panic disorder
                   ...          
189635    open wound of the nose
189636    open wound of the nose
189637    open wound of the nose
189638    open wound of the nose
189639    open wound of the nose
Name: diseases, Length: 189640, dtype: str

In [7]:
y.nunique()

713

In [8]:
print(X.dtypes.value_counts())

int64    328
Name: count, dtype: int64


In [9]:
print("Unique values in features:")
unique_values = X.nunique().sort_values()
print(unique_values.head(20))

Unique values in features:
anxiety_and_nervousness             2
depression                          2
shortness_of_breath                 2
depressive_or_psychotic_symptoms    2
sharp_chest_pain                    2
dizziness                           2
insomnia                            2
abnormal_involuntary_movements      2
chest_tightness                     2
palpitations                        2
irregular_heartbeat                 2
breathing_fast                      2
hoarse_voice                        2
sore_throat                         2
difficulty_speaking                 2
cough                               2
nasal_congestion                    2
throat_swelling                     2
diminished_hearing                  2
lump_in_throat                      2
dtype: int64


In [10]:
constant_features = [
    col for col in X.columns
    if X[col].nunique() <= 1
]

print("Number of constant features:", len(constant_features))
print(constant_features)

Number of constant features: 0
[]


In [11]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Original:")
print(y.head())

print("\nEncoded:")
print(y_encoded[:10])

Original:
0    panic disorder
1    panic disorder
2    panic disorder
3    panic disorder
4    panic disorder
Name: diseases, dtype: str

Encoded:
[485 485 485 485 485 485 485 485 485 485]


In [12]:
num_classes = len(label_encoder.classes_)
print("Number of classes:", num_classes)

Number of classes: 713


In [13]:
y_encoded

array([485, 485, 485, ..., 462, 462, 462], shape=(189640,))

In [15]:
class_counts = pd.Series(y_encoded).value_counts()

print("Number of classes:", len(class_counts))
print("Minimum samples in a class:", class_counts.min())

print("\nClasses with only 1 sample:")
print(class_counts[class_counts == 1])

Number of classes: 713
Minimum samples in a class: 1

Classes with only 1 sample:
154    1
565    1
360    1
451    1
683    1
53     1
665    1
667    1
570    1
263    1
391    1
Name: count, dtype: int64


In [16]:
print("Classes with 1 sample:", (class_counts == 1).sum())
print("Classes with < 5 samples:", (class_counts < 5).sum())
print("Classes with < 10 samples:", (class_counts < 10).sum())


Classes with 1 sample: 11
Classes with < 5 samples: 54
Classes with < 10 samples: 125


In [17]:
print("Total classes:", y.nunique())

print("Classes with >= 10 samples:",
      (class_counts >= 10).sum())

print("Classes removed if minimum = 10:",
      (class_counts < 10).sum())

Total classes: 713
Classes with >= 10 samples: 588
Classes removed if minimum = 10: 125


In [18]:
thresholds = [2, 3, 5, 10, 20, 50]

for threshold in thresholds:
    remaining = (class_counts >= threshold).sum()
    removed = (class_counts < threshold).sum()
    
    print(
        f"Minimum {threshold:>2} samples → "
        f"Remaining: {remaining:>3} | "
        f"Removed: {removed:>3}"
    )

Minimum  2 samples → Remaining: 702 | Removed:  11
Minimum  3 samples → Remaining: 690 | Removed:  23
Minimum  5 samples → Remaining: 659 | Removed:  54
Minimum 10 samples → Remaining: 588 | Removed: 125
Minimum 20 samples → Remaining: 513 | Removed: 200
Minimum 50 samples → Remaining: 414 | Removed: 299


In [19]:
min_samples = 5

class_counts = df["diseases"].value_counts()

valid_diseases = class_counts[
    class_counts >= min_samples
].index

df_filtered = df[
    df["diseases"].isin(valid_diseases)
].copy()

print("Original shape:", df.shape)
print("Filtered shape:", df_filtered.shape)

print("Original classes:", df["diseases"].nunique())
print("Remaining classes:", df_filtered["diseases"].nunique())

Original shape: (189640, 329)
Filtered shape: (189500, 329)
Original classes: 713
Remaining classes: 659


In [20]:
X = df_filtered.drop(columns=["diseases"])
y = df_filtered["diseases"]

print("X shape:", X.shape)
print("y shape:",y.shape)

X shape: (189500, 328)
y shape: (189500,)


In [34]:
feature_names = X.columns.tolist()

joblib.dump(feature_names,"../models/disease_feature_names.pkl")

print("Feature names saved successfully.")
print("Number of features:",len(feature_names))

Feature names saved successfully.
Number of features: 328


In [21]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Number of classes:", len(label_encoder.classes_))
print("y_encoded shape:", y_encoded.shape)

Number of classes: 659
y_encoded shape: (189500,)


In [29]:
joblib.dump(label_encoder,"../models/disease_label_encoder.pkl")

['../models/disease_label_encoder.pkl']

In [22]:
encoded_counts = pd.Series(y_encoded).value_counts()

print("Minimum class count:", encoded_counts.min())
print("Maximum class count:", encoded_counts.max())

Minimum class count: 5
Maximum class count: 1219


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state= 42,
    stratify=y_encoded
)

In [24]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (151600, 328)
X_test : (37900, 328)
y_train: (151600,)
y_test : (37900,)


In [25]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [26]:
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

X_train: (121280, 328)
X_val  : (30320, 328)
X_test : (37900, 328)
y_train: (121280,)
y_val  : (30320,)
y_test : (37900,)


In [27]:
joblib.dump(X_train,"../data/processed/X_train.pkl")
joblib.dump(X_val,"../data/processed/X_val.pkl")
joblib.dump(X_test,"../data/processed/X_test.pkl")

joblib.dump(y_train,"../data/processed/y_train.pkl")
joblib.dump(y_val,"../data/processed/y_val.pkl")
joblib.dump(y_test,"../data/processed/y_test.pkl")

['../data/processed/y_test.pkl']